# 第 15 章：Random Forest、交叉驗證與超參數調整

**學習目標**：
- 建立 Random Forest 分類 Pipeline。
- 使用交叉驗證降低單次切分的偶然性。
- 使用 GridSearchCV 搜尋超參數。
- 加入 `min_samples_leaf` 控制模型複雜度。
- 比較訓練與驗證 AUC，檢查過度擬合。
- 使用獨立驗證集調整分類門檻。
- 在保留測試集上評估模型。

## 學習流程

1. 建立模型資料與特徵
2. 切分訓練、驗證與測試集
3. 建立前處理與 Random Forest Pipeline
4. 定義擴充超參數網格
5. 使用三折交叉驗證搜尋
6. 檢視所有組合與過度擬合差距
7. 比較原始網格與擴充網格
8. 使用驗證集選擇 F1 門檻
9. 在測試集進行最終評估
10. 進行完整資料交叉驗證

## 1. 環境設定與建立模型資料

`build_features(data)` 會建立一列一個工作階段的資料表，目標欄位 `target` 表示是否購買。特徵沿用第 14 章，避免將直接定義目標的 `purchase` 欄位放入模型。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from common import build_features, ensure_packages, load_data

ensure_packages()
data = load_data()
model_df = build_features(data)

print(f"模型資料：{len(model_df):,} 列、{model_df.shape[1]} 欄")
print(f"購買率：{model_df['target'].mean():.2%}")

模型資料：70,000 列、15 欄
購買率：12.36%


## 2. 定義模型特徵

Random Forest 不要求數值特徵標準化，因此數值 Pipeline 只填補缺失值；類別特徵則填補後進行 One-Hot Encoding。

In [2]:
numeric_features = [
    "session_hour",
    "is_weekend",
    "page_view",
    "add_to_cart",
]
categorical_features = [
    "device",
    "traffic_source",
    "campaign",
    "experiment_group",
    "segment",
    "acquisition_channel",
]
features = numeric_features + categorical_features

X = model_df[features]
y = model_df["target"]

display(pd.DataFrame({
    "特徵": features,
    "類型": ["數值"] * len(numeric_features)
    + ["類別"] * len(categorical_features),
}))

,特徵,類型
0,session_hour,數值
1,is_weekend,數值
2,page_view,數值
3,add_to_cart,數值
4,device,類別
5,traffic_source,類別
6,campaign,類別
7,experiment_group,類別
8,segment,類別
9,acquisition_channel,類別


## 3. 切分訓練、驗證與測試資料

- 訓練集：Grid Search 與模型參數估計。
- 驗證集：選擇分類門檻。
- 測試集：最後一次評估。

原始程式在訓練資料上選門檻；本 Notebook 使用獨立驗證集，降低門檻對訓練資料過度適配的風險。

In [3]:
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=0.20,
    random_state=42,
    stratify=y_train_valid,
)

split_summary = pd.DataFrame({
    "資料集": ["訓練集", "驗證集", "測試集"],
    "筆數": [len(X_train), len(X_valid), len(X_test)],
    "正類比例": [y_train.mean(), y_valid.mean(), y_test.mean()],
})
display(split_summary)

,資料集,筆數,正類比例
0,訓練集,42000,0.123619
1,驗證集,10500,0.123619
2,測試集,17500,0.123600


## 4. 建立前處理與 Random Forest Pipeline

`Pipeline` 確保每一折交叉驗證都只使用該折的訓練資料學習填補值與類別編碼，避免驗證折資訊洩漏到前處理。

`n_jobs=1` 讓 Random Forest 本身單執行緒運作，避免 Grid Search 與模型同時平行造成資源過度使用。

In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=42, n_jobs=1)),
])

## 5. 定義超參數網格

原課程搜尋：

- `n_estimators`：森林中的樹數量。更多樹通常較穩定，但訓練較慢。
- `max_depth`：單棵樹的最大深度。越深模型越複雜。

練習再加入：

- `min_samples_leaf`：葉節點至少需要的樣本數。提高此值可讓決策邊界更平滑，有助於降低過度擬合。

共有 `2 × 2 × 3 = 12` 種組合，每種進行三折交叉驗證。

In [5]:
parameter_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 8],
    "model__min_samples_leaf": [1, 5, 10],
}

search_combinations = int(np.prod([len(values) for values in parameter_grid.values()]))
print(f"搜尋組合數：{search_combinations}")
print(f"三折交叉驗證總擬合次數：約 {search_combinations * 3} 次，另加最佳模型重訓 1 次")

搜尋組合數：12
三折交叉驗證總擬合次數：約 36 次，另加最佳模型重訓 1 次


## 6. 執行 Grid Search

使用 `roc_auc` 作為搜尋指標，因為 AUC 不依賴單一分類門檻。`return_train_score=True` 會保留每組參數的訓練 AUC，方便計算訓練／驗證差距。

In [6]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parameter_grid,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=1,
    return_train_score=True,
    refit=True,
)
grid_search.fit(X_train, y_train)

print("最佳參數：", grid_search.best_params_)
print(f"最佳交叉驗證 AUC：{grid_search.best_score_:.4f}")

最佳參數： {'model__max_depth': 5, 'model__min_samples_leaf': 10, 'model__n_estimators': 200}
最佳交叉驗證 AUC：0.6881


## 7. 整理所有搜尋結果

將 `cv_results_` 轉成易讀表格，並新增：

`AUC gap = 平均訓練 AUC − 平均驗證 AUC`

差距較大可能表示過度擬合，但不能只看 gap；也要同時看驗證分數本身與不同折的穩定程度。

In [7]:
search_results = pd.DataFrame(grid_search.cv_results_)
result_columns = [
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]
search_table = search_results[result_columns].copy()
search_table.columns = [
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "train_auc",
    "valid_auc",
    "valid_auc_std",
    "rank",
]
search_table["auc_gap"] = search_table["train_auc"] - search_table["valid_auc"]
search_table = search_table.sort_values(["rank", "auc_gap"]).reset_index(drop=True)

display(search_table.round(4))

,n_estimators,max_depth,min_samples_leaf,train_auc,valid_auc,valid_auc_std,rank,auc_gap
0,200,5,10,0.6978,0.6881,0.0070,1,0.0097
1,200,5,1,0.6980,0.6881,0.0071,2,0.0100
2,200,5,5,0.6979,0.6881,0.0070,3,0.0099
3,100,5,10,0.6972,0.6878,0.0070,4,0.0093
4,100,5,1,0.6973,0.6878,0.0071,5,0.0095
5,100,5,5,0.6972,0.6878,0.0070,6,0.0095
6,200,8,10,0.7290,0.6874,0.0070,7,0.0416
7,200,8,5,0.7337,0.6874,0.0068,8,0.0463
8,100,8,10,0.7283,0.6872,0.0072,9,0.0411
9,100,8,5,0.7329,0.6870,0.0068,10,0.0459


## 8. 最佳模型的過度擬合差距

Grid Search 的最佳索引可直接定位其平均訓練與驗證 AUC。

In [8]:
best_index = grid_search.best_index_
best_train_auc = grid_search.cv_results_["mean_train_score"][best_index]
best_valid_auc = grid_search.cv_results_["mean_test_score"][best_index]
best_auc_gap = best_train_auc - best_valid_auc

print(f"最佳模型平均訓練 AUC：{best_train_auc:.4f}")
print(f"最佳模型平均驗證 AUC：{best_valid_auc:.4f}")
print(f"訓練／驗證 AUC 差距：{best_auc_gap:.4f}")

最佳模型平均訓練 AUC：0.6978
最佳模型平均驗證 AUC：0.6881
訓練／驗證 AUC 差距：0.0097


## 9. 比較原始網格與擴充網格

原 `lesson15.py` 相當於固定 `min_samples_leaf=1`，只搜尋樹數量和最大深度。因擴充搜尋已包含這些組合，不必再重新訓練一次；可直接從同一份結果篩出原始子網格的最佳模型，再與擴充網格最佳結果比較。

In [9]:
original_grid_results = search_table.loc[
    search_table["min_samples_leaf"] == 1
].copy()
original_best = original_grid_results.sort_values(
    "valid_auc", ascending=False
).iloc[0]
expanded_best = search_table.iloc[0]

grid_comparison = pd.DataFrame([
    {
        "網格": "原始網格（leaf=1）",
        "n_estimators": original_best["n_estimators"],
        "max_depth": original_best["max_depth"],
        "min_samples_leaf": original_best["min_samples_leaf"],
        "valid_auc": original_best["valid_auc"],
        "auc_gap": original_best["auc_gap"],
    },
    {
        "網格": "擴充網格",
        "n_estimators": expanded_best["n_estimators"],
        "max_depth": expanded_best["max_depth"],
        "min_samples_leaf": expanded_best["min_samples_leaf"],
        "valid_auc": expanded_best["valid_auc"],
        "auc_gap": expanded_best["auc_gap"],
    },
])
display(grid_comparison.round(4))

,網格,n_estimators,max_depth,min_samples_leaf,valid_auc,auc_gap
0,原始網格（leaf=1）,200.0,5.0,1.0,0.6881,0.0100
1,擴充網格,200.0,5.0,10.0,0.6881,0.0097


## 10. 使用驗證集選擇分類門檻

Grid Search 選出的最佳模型已在訓練集重訓。接著使用獨立驗證集的預測機率，找出 F1 最高的門檻。測試集仍未參與任何選擇。

In [10]:
best_model = grid_search.best_estimator_
valid_probability = best_model.predict_proba(X_valid)[:, 1]

precision_curve, recall_curve, thresholds = precision_recall_curve(
    y_valid, valid_probability
)
f1_values = (
    2 * precision_curve[:-1] * recall_curve[:-1]
    / (precision_curve[:-1] + recall_curve[:-1] + 1e-12)
)
best_threshold_index = int(np.argmax(f1_values))
best_threshold = float(thresholds[best_threshold_index])

print(f"驗證集最佳 F1 門檻：{best_threshold:.4f}")
print(f"該門檻的驗證集 F1：{f1_values[best_threshold_index]:.4f}")

驗證集最佳 F1 門檻：0.1512
該門檻的驗證集 F1：0.3218


## 11. 在測試集進行最終評估

比較預設門檻 0.5 與驗證集調整門檻。AUC 使用預測機率，所以不受門檻改變影響。

In [11]:
test_probability = best_model.predict_proba(X_test)[:, 1]
test_prediction_default = (test_probability >= 0.5).astype(int)
test_prediction_tuned = (test_probability >= best_threshold).astype(int)

test_metrics = pd.DataFrame([
    {
        "門檻": "預設 0.5",
        "threshold": 0.5,
        "auc": roc_auc_score(y_test, test_probability),
        "f1": f1_score(y_test, test_prediction_default, zero_division=0),
        "precision": precision_score(y_test, test_prediction_default, zero_division=0),
        "recall": recall_score(y_test, test_prediction_default, zero_division=0),
    },
    {
        "門檻": "驗證集最佳 F1",
        "threshold": best_threshold,
        "auc": roc_auc_score(y_test, test_probability),
        "f1": f1_score(y_test, test_prediction_tuned, zero_division=0),
        "precision": precision_score(y_test, test_prediction_tuned, zero_division=0),
        "recall": recall_score(y_test, test_prediction_tuned, zero_division=0),
    },
])
display(test_metrics.round(4))

,門檻,threshold,auc,f1,precision,recall
0,預設 0.5,0.5000,0.6872,0.0000,0.0000,0.0000
1,驗證集最佳 F1,0.1512,0.6872,0.3182,0.2548,0.4235


## 12. 分類報告與混淆矩陣

In [12]:
print("調整門檻後的測試集分類報告：")
print(
    classification_report(
        y_test,
        test_prediction_tuned,
        digits=3,
        zero_division=0,
    )
)

test_confusion = confusion_matrix(y_test, test_prediction_tuned)
display(pd.DataFrame(
    test_confusion,
    index=["實際未購買", "實際購買"],
    columns=["預測未購買", "預測購買"],
))

調整門檻後的測試集分類報告：
              precision    recall  f1-score   support

           0      0.910     0.825     0.866     15337
           1      0.255     0.423     0.318      2163

    accuracy                          0.776     17500
   macro avg      0.583     0.624     0.592     17500
weighted avg      0.829     0.776     0.798     17500



,預測未購買,預測購買
實際未購買,12658,2679
實際購買,1247,916


## 13. 完整資料的三折交叉驗證

沿用原課程的最後步驟，將最佳 Pipeline 在完整資料上進行三折交叉驗證，觀察 AUC 的平均值與標準差。

這個結果可作為模型穩定性的補充描述，但因超參數已使用同一資料集搜尋過，它不是完全無偏的最終效能估計；正式流程可使用巢狀交叉驗證。

In [13]:
cv_auc_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=1,
)

print("各折 AUC：", np.round(cv_auc_scores, 4))
print(f"CV AUC 平均：{cv_auc_scores.mean():.4f}")
print(f"CV AUC 標準差：{cv_auc_scores.std():.4f}")

各折 AUC： [0.6733 0.6988 0.6847]
CV AUC 平均：0.6856
CV AUC 標準差：0.0104


## 14. 練習題：加入 `max_features`

請在小型網格中加入 Random Forest 的 `max_features`：

- `"sqrt"`：每次分裂考慮特徵數的平方根。
- `None`：每次分裂考慮全部特徵。

為控制執行時間，固定其他參數，只比較這兩種設定的三折 CV AUC 與 train/valid gap。

In [14]:
# TODO：此儲存格提供設定範例；取消註解後可自行執行額外搜尋。
exercise_grid = {
    "model__n_estimators": [grid_search.best_params_["model__n_estimators"]],
    "model__max_depth": [grid_search.best_params_["model__max_depth"]],
    "model__min_samples_leaf": [grid_search.best_params_["model__min_samples_leaf"]],
    "model__max_features": ["sqrt", None],
}

print("額外練習網格：")
for parameter, values in exercise_grid.items():
    print(f"- {parameter}: {values}")
print("此練習會額外訓練模型，請依需要建立新的 GridSearchCV 執行。")

額外練習網格：
- model__n_estimators: [200]
- model__max_depth: [5]
- model__min_samples_leaf: [10]
- model__max_features: ['sqrt', None]
此練習會額外訓練模型，請依需要建立新的 GridSearchCV 執行。


## 常見錯誤與延伸

**常見錯誤**：
- 在交叉驗證前先對完整資料填補或編碼，造成資料洩漏。
- 使用測試集選超參數或分類門檻，使最終評估過度樂觀。
- 網格設定過大，卻未估算組合數、折數與總擬合次數。
- 只看最佳 CV AUC，不檢查訓練／驗證差距與折間標準差。
- 認為訓練 AUC 越高越好；過高且差距大可能是過度擬合。
- 調整分類門檻後誤以為 AUC 也會改變。

**延伸練習**：
- 使用 RandomizedSearchCV 探索較大的參數空間。
- 繪製參數與 CV AUC／gap 的關係圖。
- 使用巢狀交叉驗證估計調參後的泛化能力。
- 比較 Logistic Regression 與 Random Forest 的 AUC、F1、訓練時間與可解釋性。

## 重點整理

- Pipeline 讓前處理在每個交叉驗證折內正確執行。
- GridSearchCV 系統性比較參數組合，但計算成本會隨組合數與折數成長。
- `min_samples_leaf` 可限制葉節點過小，幫助控制模型複雜度。
- 訓練／驗證 AUC gap 是過度擬合線索之一。
- 超參數、門檻與最終測試評估應使用不同資料角色。